## Wielowymiarowe ujęcie zróżnicowania poziomu jakości życia w województwach w Polsce
### Projekt Zaliczeniowy - Statystyczna Analiza Danych

### 0. Przygotowanie danych

In [4]:
import os
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

catalog = "./data"
csv_files = []
var_name_list = []

with os.scandir(catalog) as entries:
    for entry in entries:
        if entry.is_file() and entry.name.endswith(".csv"):
            csv_files.append(entry.path)

df_2015 = None
df_2024 = None

for file in csv_files:
    var_name = os.path.basename(file).replace(".csv", "")
    var_name_list.append(var_name)
    df = pd.read_csv(file, sep=";")

    column_2015 = [c for c in df.columns if '2015' in c][0]
    column_2024 = [c for c in df.columns if '2024' in c][0]
    df[column_2015] = pd.to_numeric(df[column_2015].astype(str).str.replace(',', '.').str.strip(), errors='coerce')
    df[column_2024] = pd.to_numeric(df[column_2024].astype(str).str.replace(',', '.').str.strip(), errors='coerce')

    temp_2015 = pd.DataFrame()
    temp_2024 = pd.DataFrame()

    temp_2015['Nazwa'] = df['Nazwa'].copy()
    temp_2015[var_name] = df[column_2015].copy()

    temp_2024['Nazwa'] = df['Nazwa'].copy()
    temp_2024[var_name] = df[column_2024].copy()

    if df_2015 is None:
        df_2015 = temp_2015
        df_2024 = temp_2024
    else:
        df_2015 = pd.merge(df_2015, temp_2015, on='Nazwa')
        df_2024 = pd.merge(df_2024, temp_2024, on='Nazwa')

df_2015 = df_2015.set_index('Nazwa')
df_2024 = df_2024.set_index('Nazwa')

Wynik: dane są przygotowane w osobnych ramkach danych z względu na aktualność danych, raz z roku 2025 i raz z najbardziej aktualnego roku możliwego z GUSu

### 1. Wprowadzenie - dane

$X1$ - Liczba zarejestrowanych podmiotów gospodarczych w przeliczeniu na 1000 mieszka **[podmioty_gospodarcze]**

$X2$ - liczby bezrobotnych w liczbie osób w wieku produkcyjnym (%) **[stopa_bezrobocia]**

$X3$ - Powierzchnia mieszkań oddanych do użytku w m2 na jednego mieszkańca (m2/os.) **[powierzchnia_mieszkaniowa]**

$X4$ - Gęstość zaludnienia (os./km2) **[gestosc_zaludnienia]**

$X5$ - Wskaźnik wykrywalności przestępstw kryminalnych (%) **[przestepstwa]** ZLE!!!!!!!!!!

$X6$ - Emisja zanieczyszczeń pyłowych na 1 km2 (emisja_pylow)

$X7$ - Udziaá parków, zieleńców i terenów zieleni osiedlowej w powierzchni ogółem (%) (udzial_zieleni_w_powierzchni)

$X8$ - Nakłady inwestycyjne na jednego mieszkańca (zł) (naklady_inwestycyjne)

$X9$ - średnie wynagrodzenie brutto (zł) (srednie_wynagordzenie)

---

### 2. Statystyka Opisowa

#### 2.1 Dodanie funkcji liczącej wszystkie statystyki

podstawowe miary statystyki opisowej dla każdej zmiennej diagnostycznej:

$\bar{x}$ – średnia arytmetyczna

$M_e$ – mediana

$S_x$ – odchylenie standardowe z próby

$V_s$ – klasyczny współczynnik zmienności

$K$ – kurtoza

$A_s$ – współczynnik asymetrii / skośność

$MIN$ – wartość minimalna wraz z nazwą województwa

$MAX$ – wartość maksymalna wraz z nazwą województwa


In [5]:
def oblicz_statystyki_opisowe(df, rok):
    stat = []
    for col in df.columns:
        s = df[col]
        stat.append({
            'Zmienna': col,
            'Rok': rok,
            'Średnia': s.mean(),
            'Mediana': s.median(),
            'Odchylenie standardowe': s.std(ddof=1),
            'Wspołczynnik zmienności': s.std(ddof=1)/s.mean() if s.mean()!=0 else 0,
            'Kurtoza': s.kurtosis(),
            'Skośność': s.skew(),
            'Max': f"{s.idxmax()}: {s.max()}",
            'Min': f"{s.idxmin()}: {s.min()}"
        })
    return pd.DataFrame(stat)

tabela1 = oblicz_statystyki_opisowe(df_2015, '2015')
tabela2 = oblicz_statystyki_opisowe(df_2024, '2024')

tabela = pd.concat([tabela1, tabela2], ignore_index=True)
tabela = tabela.sort_values(by=['Zmienna', 'Rok']).reset_index(drop=True)
tabela = tabela.set_index(['Zmienna', 'Rok'])

#### 2.2 Analiza statystyk opisowych

In [6]:
display(tabela.loc[[var_name_list[0]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna       Rok                                              
emisja_pylkow 2015     0.16     0.12                    0.18   
              2024     0.06     0.04                    0.05   

                    Wspołczynnik zmienności  Kurtoza  Skośność            Max  \
Zmienna       Rok                                                               
emisja_pylkow 2015                     1.13    14.57      3.70  ŚLĄSKIE: 0.83   
              2024                     0.95    11.65      3.19  ŚLĄSKIE: 0.25   

                                          Min  
Zmienna       Rok                              
emisja_pylkow 2015  WARMIŃSKO-MAZURSKIE: 0.04  
              2024  WARMIŃSKO-MAZURSKIE: 0.01

In [7]:
display(tabela.loc[[var_name_list[1]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna             Rok                                              
gestosc_zaludnienia 2015   126.03   116.10                   74.93   
                    2024   121.45   110.40                   72.13   

                          Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna             Rok                                                
gestosc_zaludnienia 2015                     0.59     7.12      2.43   
                    2024                     0.59     5.80      2.22   

                                     Max                        Min  
Zmienna             Rok                                              
gestosc_zaludnienia 2015  ŚLĄSKIE: 370.6            PODLASKIE: 58.9  
                    2024  ŚLĄSKIE: 347.9  WARMIŃSKO-MAZURSKIE: 55.8

In [8]:
display(tabela.loc[[var_name_list[2]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna              Rok                                              
naklady_inwestycyjne 2015  6599.65  6383.00                 1470.56   
                     2024 10780.82 10553.00                 2939.86   

                           Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna              Rok                                                
naklady_inwestycyjne 2015                     0.22     3.55      1.45   
                     2024                     0.27     9.43      2.70   

                                          Max                        Min  
Zmienna              Rok                                                  
naklady_inwestycyjne 2015  MAZOWIECKIE: 10868       ŚWIĘTOKRZYSKIE: 4624  
                     2024  MAZOWIECKIE: 20911  WARMIŃSKO-MAZURSKIE: 7741

In [9]:
display(tabela.loc[[var_name_list[3]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna              Rok                                              
podmioty_gospodarcze 2015   105.35   102.00                   19.41   
                     2024   134.76   127.00                   24.46   

                           Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna              Rok                                                
podmioty_gospodarcze 2015                     0.18    -0.96      0.31   
                     2024                     0.18    -0.18      0.64   

                                        Max                Min  
Zmienna              Rok                                        
podmioty_gospodarcze 2015  MAZOWIECKIE: 143   PODKARPACKIE: 78  
                     2024  MAZOWIECKIE: 191  PODKARPACKIE: 104

In [10]:
display(tabela.loc[[var_name_list[4]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna                   Rok                                              
powierzchnia_mieszkaniowa 2015    26.71    26.50                    1.49   
                          2024    31.80    31.90                    1.51   

                                Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna                   Rok                                                
powierzchnia_mieszkaniowa 2015                     0.06    -0.28     -0.14   
                          2024                     0.05     0.10     -0.42   

                                              Max                        Min  
Zmienna                   Rok                                                 
powierzchnia_mieszkaniowa 2015  MAZOWIECKIE: 29.5  WARMIŃSKO-MAZURSKIE: 23.9  
                          2024  MAZOWIECKIE: 34.5  WARMIŃSKO-MAZURSKIE: 28.6

In [11]:
display(tabela.loc[[var_name_list[5]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna      Rok                                              
przestepstwa 2015 49385.71 39778.00                30397.29   
             2024 47397.24 33071.00                31570.15   

                   Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna      Rok                                                
przestepstwa 2015                     0.62     0.72      1.24   
             2024                     0.67     0.34      1.14   

                                   Max               Min  
Zmienna      Rok                                          
przestepstwa 2015      ŚLĄSKIE: 117163  PODLASKIE: 17470  
             2024  MAZOWIECKIE: 119461   OPOLSKIE: 15876

In [12]:
display(tabela.loc[[var_name_list[6]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna               Rok                                              
srednie_wynagrodzenie 2015  3853.89  3790.76                  393.38   
                      2024  8156.40  7958.35                  640.67   

                            Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna               Rok                                                
srednie_wynagrodzenie 2015                     0.10     5.82      2.18   
                      2024                     0.08     3.39      1.71   

                                              Max  \
Zmienna               Rok                           
srednie_wynagrodzenie 2015   MAZOWIECKIE: 5098.55   
                      2024  MAZOWIECKIE: 10018.71   

                                                     Min  
Zmienna               Rok                                 
srednie_wynagrodzenie 2015  WARMIŃSKO-MAZURSKIE: 3495.02  
                      2024         PODKARPACKIE: 7514.69

In [13]:
display(tabela.loc[[var_name_list[7]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna          Rok                                              
stopa_bezrobocia 2015    10.82    10.50                    2.59   
                 2024     3.89     4.00                    0.96   

                       Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna          Rok                                                
stopa_bezrobocia 2015                     0.24    -0.34      0.15   
                 2024                     0.25    -1.18     -0.08   

                                             Max                 Min  
Zmienna          Rok                                                  
stopa_bezrobocia 2015  WARMIŃSKO-MAZURSKIE: 16.2  WIELKOPOLSKIE: 6.1  
                 2024          PODKARPACKIE: 5.5  WIELKOPOLSKIE: 2.3

In [14]:
display(tabela.loc[[var_name_list[8]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna                      Rok                                              
udzial_zieleni_w_powierzchni 2015     0.20     0.20                    0.17   
                             2024     0.22     0.20                    0.14   

                                   Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna                      Rok                                                
udzial_zieleni_w_powierzchni 2015                     0.83    12.05      3.26   
                             2024                     0.63    10.15      2.88   

                                            Max             Min  
Zmienna                      Rok                                 
udzial_zieleni_w_powierzchni 2015  ŚLĄSKIE: 0.8  LUBELSKIE: 0.1  
                             2024  ŚLĄSKIE: 0.7  LUBELSKIE: 0.1